# LIBERO Task Test

In [ ]:
# @title Check Colab platform
import sys
import platform

print("Python:", sys.version)
print("OS:", platform.platform())

!nvidia-smi

In [ ]:
# @title Import LeRobot and Libero
!git clone https://github.com/huggingface/lerobot.git
%cd lerobot
!pip install -e ".[libero]"

In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

project = Path(
    "/content/drive/MyDrive/Projects/humanoid-challenge"
)

assert project.exists(), f"Project not found: {project}"

print("Existing pilot files:")
for path in (project / "data/processed/pilot_01").glob("*"):
    print(path.name)

In [ ]:
from libero.libero import benchmark
suite = benchmark.get_benchmark_dict()['libero_spatial']()
task_name = "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate"

task_id = next(i for i in range(suite.n_tasks) if suite.get_task(i).name == task_name)
task = suite.get_task(task_id)

task_id, task.language

In [ ]:
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv

# retrieve a specific task
bddl_path = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

# step over the environment
env = OffScreenRenderEnv(
    bddl_file_name=bddl_path,
    camera_heights=256,
    camera_widths=256,
)

env.seed(0)
env.reset()

# for benchmarking purpose, we fix the a set of initial states
init_states = suite.get_task_init_states(task_id)
obs = env.set_init_state(init_states[0])

print("Available initial states:", len(init_states))

In [ ]:
import numpy as np

for key, value in obs.items():
    array = np.asarray(value)
    print(f"{key:35s} shape={str(array.shape):15s} dtype={array.dtype}")

low, high = env.env.action_spec

print("\nObservable size:", len(env.env.active_observables))
print("Observable modes:", env.env.observation_modalities)
print("Action dimension:", env.env.action_dim)
print("Action lower bounds:", low)
print("Action upper bounds:", high)

In [ ]:
import matplotlib.pyplot as plt

camera_keys = ["agentview_image", "robot0_eye_in_hand_image"]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, key in zip(axes, camera_keys):
    # Flip vertically for display: raw MuJoCo images can be upside down.
    ax.imshow(obs[key][::-1])
    ax.set_title(key)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
robot = env.robots[0]

print("Controller configuration:")
print(robot.controller_config)

print("\nEnd-effector position:", obs["robot0_eef_pos"])
print("End-effector quaternion:", obs["robot0_eef_quat"])
print("Gripper joint positions:", obs["robot0_gripper_qpos"])
print("Initial task success:", env.check_success())

In [ ]:
# @title Testing how to move the robot arm and record a video
import imageio.v2 as imageio
from IPython.display import Video, display

# Start from the same initial condition.
env.reset()
obs = env.set_init_state(init_states[0])

low, high = env.env.action_spec
assert env.env.action_dim == 7
print("Action limits:", low, high)
print("Gripper class:", type(env.robots[0].gripper).__name__)

# Actions are: [dx, dy, dz, rotation_x, rotation_y, rotation_z, gripper]

frames = []

def step_and_record(action, steps):
    global obs
    for _ in range(steps):
        obs, reward, done, info = env.step(action)
        frames.append(
            np.ascontiguousarray(obs["agentview_image"][::-1])
        )
        if done:
            break

# Allow the scene to settle while opening the gripper.
hold = np.zeros(7)
hold[-1] = 1.0
step_and_record(hold, 20)

start = obs["robot0_eef_pos"].copy()

rotation = hold.copy()
rotation[4] = 1.0
step_and_record(rotation, 20)

# Small positive-z command: 0.05 × 0.05 m = 2.5 mm
# target offset per control step, repeated for 10 steps.
up = hold.copy()
up[2] = 0.5
step_and_record(up, 20)

end = obs["robot0_eef_pos"].copy()

# Continue briefly with zero pose deltas.
step_and_record(hold, 20)

print("Start position:", start)
print("End position:  ", end)
print("Measured displacement (metres):", end - start)

video_path = "/content/libero_motion_check.mp4"
imageio.mimsave(video_path, frames, fps=20)
display(Video(video_path, embed=True))

In [ ]:
# @title Downlaod demonstrations from HuggingFace
%pip install -q huggingface_hub h5py

from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

repo_id = "yifengzhu-hf/LIBERO-datasets"
dataset_root = Path("/content/libero_datasets")

# Discover the filename instead of assuming its exact suffix.
api = HfApi()
revision = api.dataset_info(repo_id).sha
files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset",
    revision=revision,
)

matches = [
    name for name in files
    if name.startswith("libero_spatial/")
    and task.name in name
    and name.endswith(".hdf5")
]

assert len(matches) == 1, f"Expected one task file, found: {matches}"

demo_path = Path(hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    revision=revision,
    filename=matches[0],
    local_dir=str(dataset_root),
))

print("Downloaded:", demo_path)
print("Dataset revision:", revision)

In [ ]:
import h5py

with h5py.File(demo_path, "r") as f:
    data = f["data"]
    demo_names = sorted(
        name for name in data.keys()
        if name.startswith("demo_")
    )
    demo_name = demo_names[0]
    demo = data[demo_name]

    print("Demonstrations:", len(demo_names))
    print("Selected:", demo_name)

    for key, value in demo.items():
        print(key, getattr(value, "shape", "(group)"))

    print("\nDataset attributes:", list(data.attrs.keys()))
    print("Episode attributes:", list(demo.attrs.keys()))

In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display

with h5py.File(demo_path, "r") as f:
    demo = f[f"data/{demo_name}"]
    recorded_states = demo["states"][:]
    recorded_actions = demo["actions"][:]

env.reset()

assert recorded_states.shape[1] == len(env.get_sim_state()), (
    "Recorded state size differs from the current simulator."
)

frames = []
success_flags = []

for state in recorded_states:
    obs = env.set_init_state(state)

    # External and wrist cameras side by side.
    front = obs["agentview_image"][::-1]
    wrist = obs["robot0_eye_in_hand_image"][::-1]
    frames.append(
        np.ascontiguousarray(np.concatenate([front, wrist], axis=1))
    )

    success_flags.append(bool(env.check_success()))

video_path = "/content/libero_demo_state_playback.mp4"
imageio.mimsave(video_path, frames, fps=20)

print("Recorded actions:", recorded_actions.shape)
print("Success observed during playback:", any(success_flags))
print("Success at last recorded state:", success_flags[-1])

display(Video(video_path, embed=True))


In [ ]:
# Reset controller and simulator, then restore the demonstration's start.
env.reset()
obs = env.set_init_state(recorded_states[0])

frames = []
success_flags = []

# Do not add settling actions: they would change the recorded starting state.
for action in recorded_actions:
    obs, reward, done, info = env.step(action)

    front = obs["agentview_image"][::-1]
    wrist = obs["robot0_eye_in_hand_image"][::-1]
    frames.append(
        np.ascontiguousarray(np.concatenate([front, wrist], axis=1))
    )

    success_flags.append(bool(env.check_success()))

    if done:
        break

video_path = "/content/libero_demo_action_replay.mp4"
imageio.mimsave(video_path, frames, fps=20)

print("Actions executed:", len(frames), "/", len(recorded_actions))
print("Success reached:", any(success_flags))
print("Success at end:", success_flags[-1])

display(Video(video_path, embed=True))

## Example retargeting: geometry-based grasp setup

This section currently tests an open-gripper approach, not the completed human-motion replay. The stage-selection cell defines where to move; the following execution cell performs those stages and records both cameras.


In [ ]:
import numpy as np

# step over the environment
env = OffScreenRenderEnv(
    bddl_file_name=bddl_path,
    camera_heights=256,
    camera_widths=256,
    horizon=5000,
    ignore_done=True,
)

ob = env.reset()

# Our OffScreenRenderEnv wraps the underlying task environment.
task_env = env.env
sim = task_env.sim

print("End-effector position:", ob["robot0_eef_pos"])
print("End-effector quaternion:", ob["robot0_eef_quat"])

In [ ]:
object_body_ids = {}

for body_id in range(sim.model.nbody):
    name = sim.model.body_id2name(body_id)

    if name and ("akita_black_bowl_1" in name or "plate_1" in name):
        object_body_ids[name] = body_id
        print(name, "| world position:", np.round(sim.data.body_xpos[body_id], 4),)

assert object_body_ids, "No matching bodies found."

In [ ]:
for body_name, body_id in object_body_ids.items():
    first_geom = int(sim.model.body_geomadr[body_id])
    number_of_geoms = int(sim.model.body_geomnum[body_id])

    print(f"\nBody: {body_name}")

    for geom_id in range(first_geom, first_geom + number_of_geoms):
        print({
            "name": sim.model.geom_id2name(geom_id),
            "type": int(sim.model.geom_type[geom_id]),
            "size_parameters": np.round(sim.model.geom_size[geom_id], 4).tolist(),
            "world_position": np.round(sim.data.geom_xpos[geom_id], 4).tolist(),
        })

In [ ]:
from itertools import product
import numpy as np

def collision_box_bounds(body_name):
    body_id = sim.model.body_name2id(body_name)

    first = int(sim.model.body_geomadr[body_id])
    count = int(sim.model.body_geomnum[body_id])

    # Eight combinations: (-1,-1,-1), (-1,-1,+1), ...
    corner_signs = np.array(
        list(product([-1, 1], repeat=3)),
        dtype=float,
    )

    world_corners = []

    for geom_id in range(first, first + count):
        # In MuJoCo, geometry type 6 is a box.
        if int(sim.model.geom_type[geom_id]) != 6:
            continue

        # Exclude geometries that have no collision masks.
        if (sim.model.geom_contype[geom_id] == 0 and sim.model.geom_conaffinity[geom_id] == 0):
            continue

        half_sizes = sim.model.geom_size[geom_id]
        position = sim.data.geom_xpos[geom_id]
        rotation = sim.data.geom_xmat[geom_id].reshape(3, 3)

        local_corners = corner_signs * half_sizes
        corners = local_corners @ rotation.T + position

        world_corners.append(corners)

    assert world_corners, f"No collision boxes found for {body_name}"

    points = np.concatenate(world_corners, axis=0)

    return {
        "min": points.min(axis=0),
        "max": points.max(axis=0),
    }


bounds = {}

for name in ["akita_black_bowl_1_main", "plate_1_main"]:
    bounds[name] = collision_box_bounds(name)

    lower = bounds[name]["min"]
    upper = bounds[name]["max"]

    print(f"\n{name}")
    print("World minimum [m]:", np.round(lower, 4))
    print("World maximum [m]:", np.round(upper, 4))
    print("World extent [mm]:", np.round((upper - lower) * 1000, 1))

In [ ]:
bowl_bounds = bounds["akita_black_bowl_1_main"]
lower = bowl_bounds["min"]
upper = bowl_bounds["max"]

bowl_center_xy = (lower[:2] + upper[:2]) / 2
bowl_top_z = upper[2]

# Direction from the bowl centre toward the current end-effector.
eef_position = np.asarray(ob["robot0_eef_pos"])

direction_xy = eef_position[:2] - bowl_center_xy
distance = np.linalg.norm(direction_xy)
assert distance > 1e-6, "Cannot determine a horizontal approach direction."

direction_xy = direction_xy / distance

# Initial geometric estimates—not calibrated grasp parameters.
outer_radius = np.mean(upper[:2] - lower[:2]) / 2
rim_radius = outer_radius - 0.006
depth_below_top = 0.010

candidate_contact = np.array([
    *(bowl_center_xy + rim_radius * direction_xy),
    bowl_top_z - depth_below_top,
])

print("Bowl centre XY:", bowl_center_xy)
print("Candidate contact location:", candidate_contact)

In [ ]:
gripper = task_env.robots[0].gripper

print("Important sites:")
print(gripper.important_sites)

print("\nImportant geometries:")
print(gripper.important_geoms)

print("\nCurrent end-effector position:")
print(ob["robot0_eef_pos"])

print("\nCurrent gripper joint positions:")
print(ob["robot0_gripper_qpos"])

print("\nAvailable gripper-related sites:")

for site_id in range(sim.model.nsite):
    name = sim.model.site_id2name(site_id)

    if name and any(word in name.lower() for word in ["gripper", "grip", "eef", "finger"]):
        print(name, "| world position:", np.round(sim.data.site_xpos[site_id], 4),)

controller = task_env.robots[0].controller

eef_name = getattr(controller, "eef_name", None)
print("Controller end-effector site:", eef_name)

if eef_name is not None:
    site_id = sim.model.site_name2id(eef_name)
    print("Controlled site position:", sim.data.site_xpos[site_id],)

In [ ]:
site_id = sim.model.site_name2id(eef_name)

grip_position = sim.data.site_xpos[site_id].copy()
grip_rotation = sim.data.site_xmat[site_id].reshape(3, 3).copy()

pad_positions = {}

for side in ["left", "right"]:
    geom_name = gripper.important_geoms[f"{side}_fingerpad"][0]
    geom_id = sim.model.geom_name2id(geom_name)

    pad_position = sim.data.geom_xpos[geom_id].copy()
    pad_positions[side] = pad_position

    # Express the pad's offset in the controlled site's coordinates.
    local_offset = grip_rotation.T @ (pad_position - grip_position)

    print(f"\n{side} pad: {geom_name}")
    print("World position [m]:", np.round(pad_position, 4))
    print("Offset in grip frame [mm]:",np.round(local_offset * 1000, 2))
    print("Geometry type:", int(sim.model.geom_type[geom_id]))
    print("Size parameters [m]:", sim.model.geom_size[geom_id])

In [ ]:
pad_midpoint = (pad_positions["left"] + pad_positions["right"]) / 2
separation = pad_positions["right"] - pad_positions["left"]
midpoint_offset_local = grip_rotation.T @ (pad_midpoint - grip_position)
closing_axis_local = grip_rotation.T @ (separation / np.linalg.norm(separation))

print("\nPad midpoint offset in grip frame [mm]:", np.round(midpoint_offset_local * 1000, 2))
print("Finger separation direction in grip frame:", np.round(closing_axis_local, 3))
print("Pad centre-to-centre distance [mm]:",round(np.linalg.norm(separation) * 1000, 2))

In [ ]:
# Local X: horizontal, pointing radially from the bowl toward the side where we selected the grasp.
x_axis = np.array([direction_xy[0], direction_xy[1], 0.0,])

# Local Z: pointing down toward the table.
z_axis = np.array([0.0, 0.0, -1.0])

# Complete a right-handed coordinate system.
y_axis = np.cross(z_axis, x_axis)
y_axis /= np.linalg.norm(y_axis)

target_rotation = np.column_stack([x_axis, y_axis, z_axis])

assert np.allclose(target_rotation.T @ target_rotation, np.eye(3))
assert np.isclose(np.linalg.det(target_rotation), 1.0)

print("Target rotation matrix:")
print(np.round(target_rotation, 3))

In [ ]:
# Use the measured offset from the preceding cell.
pad_offset_local = midpoint_offset_local.copy()

target_grip_position = (candidate_contact - target_rotation @ pad_offset_local)

# First approach target: pad midpoint 8 cm above the candidate.
# This clearance is an initial choice, not a collision guarantee.
approach_pad_position = (candidate_contact + np.array([0.0, 0.0, 0.080]))

approach_grip_position = (approach_pad_position - target_rotation @ pad_offset_local)

print("Candidate pad midpoint [m]:",np.round(candidate_contact, 4))
print("Candidate controlled-site position [m]:", np.round(target_grip_position, 4))
print("Approach controlled-site position [m]:", np.round(approach_grip_position, 4))

In [ ]:
reconstructed_pad_position = (target_grip_position + target_rotation @ pad_offset_local)
assert np.allclose(reconstructed_pad_position, candidate_contact)
print("Pad-offset calculation verified.")

In [ ]:
from scipy.spatial.transform import Rotation
import imageio.v2 as imageio
from IPython.display import Video, display

controller = task_env.robots[0].controller

# Confirm the scaling used below matches this environment.
assert controller.use_delta
assert np.allclose(controller.input_max, 1)
assert np.allclose(controller.input_min, -1)
assert np.allclose(controller.output_max, [0.05, 0.05, 0.05, 0.5, 0.5, 0.5])
assert np.allclose(controller.output_min, [-0.05, -0.05, -0.05, -0.5, -0.5, -0.5])

site_id = sim.model.site_name2id(controller.eef_name)

def current_pose():
    position = sim.data.site_xpos[site_id].copy()
    rotation = (sim.data.site_xmat[site_id].reshape(3, 3).copy())
    return position, rotation


def pose_errors(target_position, target_orientation):
    position, rotation = current_pose()
    position_error = target_position - position
    rotation_error = Rotation.from_matrix(target_orientation @ rotation.T).as_rotvec()
    return position_error, rotation_error


def approach_action(target_position, target_orientation):
    position_error, rotation_error = pose_errors(target_position, target_orientation)

    action = np.zeros(7)

    # Limit each requested translation component to 1 mm.
    action[:3] = np.clip(position_error / 0.05, -0.02, 0.02)

    # Limit each requested rotation component to 0.025 radians.
    action[3:6] = np.clip(rotation_error / 0.5, -0.05, 0.05)

    action[6] = -1.0  # Keep opening the fingers.
    return action

In [ ]:
# @title Select initial approach stages
#start_position, _ = current_pose()

#assert start_position[2] > approach_grip_position[2], (
#    "Expected a fresh starting pose above the approach height. "
#    "Do not rerun the reset unless you also recalculate all geometry targets."
#)

#above_rim = approach_grip_position.copy()
#above_rim[2] = start_position[2]

#stages = [
#    ("Open and orient", start_position.copy()),
#    ("Move above rim", above_rim),
#    ("Lower to approach height", approach_grip_position.copy()),
#]

#print("Starting grip-site position:", start_position)
#print("Approach grip-site target:", approach_grip_position)
#print("Stages:", [name for name, _ in stages])

inspection_pad_position = candidate_contact.copy()
inspection_pad_position[2] = bowl_top_z

inspection_grip_position = (inspection_pad_position - target_rotation @ pad_offset_local)

stages = [
    ("Descend to rim inspection pose", inspection_grip_position),
]

position, _ = current_pose()

print("Requested vertical change [mm]:",(inspection_grip_position[2] - position[2]) * 1000)


In [ ]:
current_bowl_top = collision_box_bounds("akita_black_bowl_1_main")["max"][2]

inspection_pad_position = candidate_contact.copy()
inspection_pad_position[2] = current_bowl_top - 0.005

inspection_grip_position = (inspection_pad_position - target_rotation @ pad_offset_local)

stages = [("Descend to 20 mm above rim", inspection_grip_position.copy())]

# Verify the target before executing any movement.
expected_pad_position = (
    stages[0][1] + target_rotation @ pad_offset_local
)

print("Stage:", stages[0][0])
print("Commanded pad clearance [mm]:",(expected_pad_position[2] - current_bowl_top) * 1000)
print("Requested grip-site displacement [mm]:", (stages[0][1] - current_pose()[0]) * 1000)

In [ ]:
approach_frames = []
approach_results = []
rotation_debug = []

for stage_name, target_position in stages:
    stable_steps = 0
    reached = False

    for step_index in range(600):  # Up to 30 simulated seconds.
        action = approach_action(target_position, target_rotation)
        ob, reward, done, info = env.step(action)

        external_view = ob["agentview_image"][::-1].copy()
        wrist_view = ob["robot0_eye_in_hand_image"][::-1].copy()

        # Put them next to each other: external on left, wrist on right.
        combined_view = np.concatenate([external_view, wrist_view], axis=1)

        approach_frames.append(combined_view)

        position_error, rotation_error = pose_errors(target_position, target_rotation)
        position_error_m = np.linalg.norm(position_error)
        rotation_error_rad = np.linalg.norm(rotation_error)
        rotation_debug.append({
            "stage": stage_name,
            "step": step_index,
            "error_deg": float(np.rad2deg(rotation_error_rad)),
            "rotation_action": action[3:6].copy(),
        })

        if step_index % 20 == 0:
            print(f"{stage_name} | step {step_index} | position error: {position_error_m * 1000:.2f} mm | rotation error: {np.rad2deg(rotation_error_rad):.1f}°")

        within_tolerance = (position_error_m < 0.003 and rotation_error_rad < np.deg2rad(3))
        stable_steps = (stable_steps + 1 if within_tolerance else 0)

        # Require stable tracking and at least 1 second per stage.
        reached = stable_steps >= 10 and step_index >= 19

        if done or reached:
            break

    result = {
        "stage": stage_name,
        "reached": reached,
        "position_error_mm": round(position_error_m * 1000, 2),
        "rotation_error_deg": round(np.rad2deg(rotation_error_rad), 2),
    }

    approach_results.append(result)
    print(result)

    if done:
        print("Stopped because the environment returned done=True.")
        break

    if not reached:
        print("Stopped because this stage used its full step allowance.")
        break

In [ ]:
from pathlib import Path

output_dir = project / "outputs/human_retargeting"
output_dir.mkdir(parents=True, exist_ok=True)

video_path = output_dir / "rim_approach.mp4"

imageio.mimsave(
    str(video_path),
    approach_frames,
    fps=20,
)

display(Video(str(video_path), embed=True))

In [ ]:
import matplotlib.pyplot as plt

# `ob` contains the observation returned by the last simulation step.
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, key, title in zip(
    axes,
    ["agentview_image", "robot0_eye_in_hand_image"],
    ["External view", "Wrist view"],
):
    ax.imshow(ob[key][::-1])
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Recompute the bowl top from its current collision geometry.
current_bowl_bounds = collision_box_bounds("akita_black_bowl_1_main")
current_bowl_top = current_bowl_bounds["max"][2]

pad_centres = []

for side in ["left", "right"]:
    name = gripper.important_geoms[f"{side}_fingerpad"][0]
    geom_id = sim.model.geom_name2id(name)

    centre = sim.data.geom_xpos[geom_id].copy()
    pad_centres.append(centre)

    print(
        f"{side} pad centre above bowl top: "
        f"{(centre[2] - current_bowl_top) * 1000:.2f} mm"
    )

actual_pad_midpoint = np.mean(pad_centres, axis=0)

print(
    "Actual pad midpoint above bowl top:",
    round((actual_pad_midpoint[2] - current_bowl_top) * 1000, 2),
    "mm",
)

print(
    "Grip-site distance from last commanded target:",
    round(
        np.linalg.norm(current_pose()[0] - target_position) * 1000,
        2,
    ),
    "mm",
)

In [ ]:
# Hold the actual current wrist pose during closure.
hold_position, hold_rotation = current_pose()

bowl_body_id = sim.model.body_name2id("akita_black_bowl_1_main")
bowl_position_before = sim.data.body_xpos[bowl_body_id].copy()

closure_frames = []

for step_index in range(20):  # 1 second at 20 Hz
    action = approach_action(hold_position, hold_rotation)

    # Override the opening command supplied by approach_action.
    action[6] = 1.0

    ob, reward, done, info = env.step(action)

    closure_frames.append(
        np.concatenate(
            [ob["agentview_image"][::-1], ob["robot0_eye_in_hand_image"][::-1],], axis=1).copy()
    )

    if done:
        print("Environment ended during closure.")
        break

bowl_displacement = (sim.data.body_xpos[bowl_body_id] - bowl_position_before)

print("Bowl displacement [mm]:", bowl_displacement * 1000)
print("Gripper joint positions:", ob["robot0_gripper_qpos"])

In [ ]:
video_path = output_dir / "rim_closure.mp4"
imageio.mimsave(str(video_path), closure_frames, fps=20)
display(Video(str(video_path), embed=True))

In [ ]:
lift_start_position, lift_rotation = current_pose()
lift_target = lift_start_position + np.array([0.0, 0.0, 0.030])

bowl_body_id = sim.model.body_name2id("akita_black_bowl_1_main")
bowl_start_position = sim.data.body_xpos[bowl_body_id].copy()

lift_frames = []
stable_steps = 0
reached = False

for step_index in range(600):
    action = approach_action(lift_target, lift_rotation)
    action[6] = 1.0  # Maintain the grasp.

    ob, reward, done, info = env.step(action)

    lift_frames.append(
        np.concatenate(
            [ob["agentview_image"][::-1],ob["robot0_eye_in_hand_image"][::-1],],axis=1).copy()
    )

    position_error, rotation_error = pose_errors(
        lift_target, lift_rotation
    )

    within_tolerance = (
        np.linalg.norm(position_error) < 0.003
        and np.linalg.norm(rotation_error) < np.deg2rad(3)
    )
    stable_steps = stable_steps + 1 if within_tolerance else 0
    reached = stable_steps >= 10

    if done or reached:
        break

eef_rise = current_pose()[0][2] - lift_start_position[2]
bowl_rise = sim.data.body_xpos[bowl_body_id][2] - bowl_start_position[2]

print("Lift target reached:", reached)
print("Environment ended:", done)
print(f"Gripper rise: {eef_rise * 1000:.1f} mm")
print(f"Bowl body rise: {bowl_rise * 1000:.1f} mm")

video_path = output_dir / "rim_grasp_lift_test.mp4"
imageio.mimsave(str(video_path), lift_frames, fps=20)
display(Video(str(video_path), embed=True))

In [ ]:
hold_position, hold_rotation = current_pose()

bowl_before = sim.data.body_xpos[bowl_body_id].copy()
eef_before = hold_position.copy()
hold_frames = []

for _ in range(40):  # 2 seconds at 20 Hz
    action = approach_action(hold_position, hold_rotation)
    action[6] = 1.0

    ob, reward, done, info = env.step(action)

    hold_frames.append(
        np.concatenate(
            [
                ob["agentview_image"][::-1],
                ob["robot0_eye_in_hand_image"][::-1],
            ],
            axis=1,
        ).copy()
    )

    if done:
        break

bowl_after = sim.data.body_xpos[bowl_body_id].copy()
eef_after, _ = current_pose()

relative_change = (
    (bowl_after - eef_after)
    - (bowl_before - eef_before)
)

print("Environment ended:", done)
print("Bowl movement [mm]:", (bowl_after - bowl_before) * 1000)
print(
    "Bowl movement relative to gripper [mm]:",
    relative_change * 1000,
)

video_path = output_dir / "grasp_hold_test.mp4"
imageio.mimsave(str(video_path), hold_frames, fps=20)
display(Video(str(video_path), embed=True))

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt

pilot_dir = project / "data/processed/pilot_01"
human_trajectory = pd.read_csv(pilot_dir / "marker_smoothed.csv")
annotation = json.loads((pilot_dir / "human_events.json").read_text())
events = annotation["events"]

transfer_start = events["grasp_established"]["time_s"]
transfer_end = events["placement_contact"]["time_s"]

# Include the annotated endpoints exactly.
sample_times = human_trajectory["seconds"].to_numpy()

assert sample_times[0] <= transfer_start < transfer_end <= sample_times[-1]
assert np.all(np.diff(sample_times) > 0)

transfer_times = np.concatenate([
    [transfer_start],
    sample_times[(sample_times > transfer_start)& (sample_times < transfer_end)],
    [transfer_end],
])

human_positions = np.column_stack([
    np.interp(transfer_times, sample_times, human_trajectory[column].to_numpy())
    for column in ["x_m", "y_m", "height_m"]
])

human_displacement = human_positions - human_positions[0]

print("Horizontal endpoint displacement [mm]:", human_displacement[-1, :2] * 1000)
print("Maximum marker rise above grasp [mm]:", human_displacement[:, 2].max() * 1000)
print("Marker height change at placement [mm]:", human_displacement[-1, 2] * 1000)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(transfer_times - transfer_start, human_displacement * 1000, label=["X", "Y", "Height"])
ax.set_xlabel("Time since human grasp [s]")
ax.set_ylabel("Marker displacement [mm]")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
current_position, current_rotation = current_pose()

# How far the wrist has risen since the grasp test began.
current_rise = current_position[2] - lift_start_position[2]

human_height = human_displacement[:, 2]
peak_index = int(np.argmax(human_height))

# Find the first ascending crossing of the current robot rise.
crossings = np.flatnonzero(
    human_height[:peak_index + 1] >= current_rise
)

assert current_rise > 0
assert len(crossings), "No matching height on the human lifting arc."

join_index = int(crossings[0])

remaining_human = human_positions[join_index:].copy()
remaining_times = transfer_times[join_index:].copy()

print(
    "Join time after human grasp [s]:",
    round(remaining_times[0] - transfer_start, 3),
)
print(
    "Human marker rise at join [mm]:",
    round(human_height[join_index] * 1000, 1),
)

In [ ]:
plate_bounds = collision_box_bounds("plate_1_main")
bowl_bounds_now = collision_box_bounds("akita_black_bowl_1_main")

plate_center_xy = (
    plate_bounds["min"][:2] + plate_bounds["max"][:2]
) / 2

# Use the bowl collision-envelope centre for horizontal alignment.
bowl_center_xy_now = (
    bowl_bounds_now["min"][:2] + bowl_bounds_now["max"][:2]
) / 2

robot_destination = current_position.copy()

# Shift the held bowl's centre above the plate centre.
robot_destination[:2] += plate_center_xy - bowl_center_xy_now

# End with the bowl bottom 20 mm above the plate's highest geometry.
# Assumes the bowl keeps approximately its current orientation.
desired_bowl_bottom = plate_bounds["max"][2] + 0.020
robot_destination[2] += (
    desired_bowl_bottom - bowl_bounds_now["min"][2]
)

human_delta_xy = remaining_human[-1, :2] - remaining_human[0, :2]
robot_delta_xy = robot_destination[:2] - current_position[:2]

assert np.linalg.norm(human_delta_xy) > 1e-6

angle = (
    np.arctan2(robot_delta_xy[1], robot_delta_xy[0])
    - np.arctan2(human_delta_xy[1], human_delta_xy[0])
)

rotation_xy = np.array([
    [np.cos(angle), -np.sin(angle)],
    [np.sin(angle),  np.cos(angle)],
])

horizontal_scale = (
    np.linalg.norm(robot_delta_xy)
    / np.linalg.norm(human_delta_xy)
)

u = (
    (remaining_times - remaining_times[0])
    / (remaining_times[-1] - remaining_times[0])
)

human_straight_path = (
    (1 - u[:, None]) * remaining_human[0]
    + u[:, None] * remaining_human[-1]
)

human_deviation = remaining_human - human_straight_path

robot_deviation = human_deviation.copy()
robot_deviation[:, :2] = (
    human_deviation[:, :2] @ rotation_xy.T
) * horizontal_scale

robot_straight_path = (
    (1 - u[:, None]) * current_position
    + u[:, None] * robot_destination
)

candidate_path = robot_straight_path + robot_deviation

assert np.allclose(candidate_path[0], current_position)
assert np.allclose(candidate_path[-1], robot_destination)

print("Horizontal scale:", round(horizontal_scale, 3))
print("Candidate destination [m]:", robot_destination)

In [ ]:
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")

ax.plot(*candidate_path.T, label="Retargeted human path")
ax.plot(
    *robot_straight_path.T,
    "--",
    label="Straight connection",
)
ax.scatter(*current_position, label="Current wrist")
ax.scatter(*robot_destination, label="Hover destination")

ax.set_xlabel("World X [m]")
ax.set_ylabel("World Y [m]")
ax.set_zlabel("World Z [m]")
ax.legend()
plt.show()

In [ ]:
actual_position, _ = current_pose()

bowl_now = collision_box_bounds("akita_black_bowl_1_main")
plate_now = collision_box_bounds("plate_1_main")

print("Actual wrist position [m]:", actual_position)
print("Path start [m]:", candidate_path[0])
print(
    "Start mismatch [mm]:",
    np.linalg.norm(candidate_path[0] - actual_position) * 1000,
)

print("Current bowl bottom Z [m]:", bowl_now["min"][2])
print("Current bowl top Z [m]:", bowl_now["max"][2])
print("Current plate top Z [m]:", plate_now["max"][2])

print("Horizontal scale:", horizontal_scale)
print("Path maximum wrist Z [m]:", candidate_path[:, 2].max())

# Estimate destination clearance assuming the held bowl
# maintains its current offset and orientation relative to the wrist.
predicted_final_bowl_bottom = (
    bowl_now["min"][2]
    + candidate_path[-1, 2]
    - actual_position[2]
)

print(
    "Predicted bowl-bottom clearance above plate [mm]:",
    (predicted_final_bowl_bottom - plate_now["max"][2]) * 1000,
)

In [ ]:
import time

# Reduce redundant targets, keeping points about 5 mm apart.
waypoints = [candidate_path[0].copy()]

for point in candidate_path[1:]:
    if np.linalg.norm(point - waypoints[-1]) >= 0.005:
        waypoints.append(point.copy())

if not np.allclose(waypoints[-1], candidate_path[-1]):
    waypoints.append(candidate_path[-1].copy())

transfer_waypoints = np.asarray(waypoints)
transfer_orientation = current_pose()[1].copy()

assert np.linalg.norm(
    current_pose()[0] - transfer_waypoints[0]
) < 0.005, "The robot moved since path generation; stop and rebuild the path."

transfer_frames = []
transfer_log = []
transfer_completed = True
total_steps = 0
started = time.monotonic()

# Monitor changes in the bowl's position relative to the wrist.
initial_bowl_offset = (
    sim.data.body_xpos[bowl_body_id].copy()
    - current_pose()[0]
)

for waypoint_index, target in enumerate(transfer_waypoints[1:], 1):
    reached = False
    stop_reason = None

    for _ in range(200):
        action = approach_action(target, transfer_orientation)
        action[6] = 1.0

        ob, reward, done, info = env.step(action)
        total_steps += 1

        position, _ = current_pose()
        position_error, rotation_error = pose_errors(
            target, transfer_orientation
        )

        bowl_offset = (
            sim.data.body_xpos[bowl_body_id] - position
        )
        relative_drift = np.linalg.norm(
            bowl_offset - initial_bowl_offset
        )

        transfer_log.append({
            "step": total_steps,
            "waypoint": waypoint_index,
            "position_error_m": float(np.linalg.norm(position_error)),
            "bowl_relative_drift_m": float(relative_drift),
        })

        # Record at 10 fps while controlling at 20 Hz.
        if total_steps % 2 == 0:
            transfer_frames.append(
                np.concatenate([
                    ob["agentview_image"][::-1],
                    ob["robot0_eye_in_hand_image"][::-1],
                ], axis=1).copy()
            )

        if done:
            stop_reason = "Environment ended"
        elif relative_drift > 0.015:
            stop_reason = "Bowl shifted over 15 mm relative to wrist"
        elif total_steps >= 4000:
            stop_reason = "Total step limit reached"

        if stop_reason:
            break

        reached = (
            np.linalg.norm(position_error) < 0.003
            and np.linalg.norm(rotation_error) < np.deg2rad(3)
        )
        if reached:
            break

    if stop_reason or not reached:
        transfer_completed = False
        print("Stopped:", stop_reason or "Waypoint tracking timed out")
        print("Waypoint:", waypoint_index)
        break

    if waypoint_index % 10 == 0:
        print(f"Reached waypoint {waypoint_index}/{len(transfer_waypoints)-1}")

video_path = output_dir / "human_transfer_hover.mp4"

if transfer_frames:
    imageio.mimsave(str(video_path), transfer_frames, fps=10)
    display(Video(str(video_path), embed=True))

(output_dir / "human_transfer_hover_log.json").write_text(
    json.dumps(transfer_log, indent=2)
)

print("Transfer completed:", transfer_completed)
print("Simulation steps:", total_steps)
print("Task success:", bool(env.check_success()))

In [ ]:
# Estimate a placement target from the current geometry.
bowl_now = collision_box_bounds("akita_black_bowl_1_main")
plate_now = collision_box_bounds("plate_1_main")

place_position, place_rotation = current_pose()
gap = bowl_now["min"][2] - plate_now["max"][2]

assert -0.005 < gap < 0.050, (
    f"Unexpected bowl/plate gap: {gap * 1000:.1f} mm"
)

# Lower toward the plate top, leaving 1 mm nominal clearance.
place_position[2] -= max(0.0, gap - 0.001)

placement_frames = []

def placement_step(target, grip):
    global ob

    action = approach_action(target, place_rotation)
    action[6] = grip
    ob, reward, done, info = env.step(action)

    placement_frames.append(
        np.concatenate([
            ob["agentview_image"][::-1],
            ob["robot0_eye_in_hand_image"][::-1],
        ], axis=1).copy()
    )

    if done:
        raise RuntimeError("Environment ended during placement.")


# Lower while maintaining the grasp.
reached = False
for _ in range(600):
    placement_step(place_position, 1.0)
    position_error, rotation_error = pose_errors(place_position, place_rotation)

    if (np.linalg.norm(position_error) < 0.002 and np.linalg.norm(rotation_error) < np.deg2rad(3)):
        reached = True
        break

if not reached:
    raise RuntimeError("Placement target not reached; gripper remains closed.")

# Open at the reached pose.
release_position, _ = current_pose()

for _ in range(30):
    placement_step(release_position, -1.0)

# Withdraw vertically with the fingers open.
retract_position = release_position + np.array([0.0, 0.0, 0.080])

retracted = False
for _ in range(600):
    placement_step(retract_position, -1.0)

    if np.linalg.norm(current_pose()[0] - retract_position) < 0.003:
        retracted = True
        break

# Allow the bowl to settle while holding the actual wrist position.
settle_position, _ = current_pose()
success_flags = []

for _ in range(40):
    placement_step(settle_position, -1.0)
    success_flags.append(bool(env.check_success()))

video_path = output_dir / "human_transfer_placement.mp4"
imageio.mimsave(str(video_path), placement_frames, fps=20)
display(Video(str(video_path), embed=True))

print("Retraction reached:", retracted)
print("Final task success:", success_flags[-1])
print("Success throughout last second:", all(success_flags[-20:]))

In [ ]:
# Show the latest camera observations.
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, key, title in zip(
    axes,
    ["agentview_image", "robot0_eye_in_hand_image"],
    ["External view", "Wrist view"],
):
    ax.imshow(ob[key][::-1])
    ax.set_title(title)
    ax.axis("off")

plt.show()

# Identify geometries belonging to each object by their names.
def object_geom_ids(prefix):
    return {
        geom_id
        for geom_id in range(sim.model.ngeom)
        if (sim.model.geom_id2name(geom_id) or "").startswith(prefix)
    }

bowl_geoms = object_geom_ids("akita_black_bowl_1_")
plate_geoms = object_geom_ids("plate_1_")

bowl_plate_contacts = []

for index in range(sim.data.ncon):
    contact = sim.data.contact[index]
    g1, g2 = int(contact.geom1), int(contact.geom2)

    if (
        (g1 in bowl_geoms and g2 in plate_geoms)
        or (g2 in bowl_geoms and g1 in plate_geoms)
    ):
        bowl_plate_contacts.append({
            "geom1": sim.model.geom_id2name(g1),
            "geom2": sim.model.geom_id2name(g2),
            "distance_mm": float(contact.dist * 1000),
        })

print("Bowl–plate contacts:", bowl_plate_contacts)
print("Current task success:", bool(env.check_success()))

actual_position, _ = current_pose()
print(
    "Distance from transfer endpoint [mm]:",
    np.linalg.norm(actual_position - candidate_path[-1]) * 1000,
)

In [ ]:
result = {
    "task_success": bool(env.check_success()),
    "bowl_plate_contacts": bowl_plate_contacts,
    "endpoint_error_mm": float(
        np.linalg.norm(current_pose()[0] - candidate_path[-1]) * 1000
    ),
    "release_verified": False,  # Set True only after observing release.
}

output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "pilot_transfer_result.json").write_text(
    json.dumps(result, indent=2)
)

print("Saved:", output_dir / "pilot_transfer_result.json")